In [ ]:
!pip install ultralytics pandas matplotlib -q
!git clone https://github.com/Dinoman67/sonarvision.git
%cd sonarvision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 6.0 MB/s eta 0:00:00
Cloning into 'sonarvision'...
remote: Enumerating objects: 4080, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4080 (delta 1), reused 3 (delta 1), pack-reused 4072 (from 2)
Receiving objects: 100% (4080/4080), 751.90 MiB | 37.11 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (7115/7115), done.
/content/sonarvision
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
import os

# Ensure we are in the correct directory to unzip h8.zip if it's in /content/
# The h8.zip is located at /content/h8.zip
# The current directory is /content/sonarvision, so we need to specify the full path.
!unzip -q /content/h8.zip -d /content/

# Fix data.yaml path for Colab
!sed -i 's|path:.*|path: /content/h8|' /content/h8/data.yaml

DATA = '/content/h8/data.yaml'
print(f'\nDataset ready: {DATA}')

# Verify
import yaml
from pathlib import Path
with open(DATA) as f:
    cfg = yaml.safe_load(f)
train_imgs = list(Path(cfg['path'], 'images', 'train').glob('*'))
val_imgs = list(Path(cfg['path'], 'images', 'val').glob('*'))
print(f'Train: {len(train_imgs)} images')
print(f'Val: {len(val_imgs)} images')
print(f'Classes: {cfg["names"]}')


Dataset ready: /content/h8/data.yaml
Train: 3110 images
Val: 438 images
Classes: ['marine_debris']


In [ ]:
import sys
from ultralytics import YOLO
from models.sss_custom_modules import (
    PConv, FasterBlock, FastC2f, GhostConv,
    SEBlock, CBAM, WaveletConv, LocalContrastEnhance
)
from models.build_sss_models import (
    build_model, build_model_a, build_model_b,
    build_model_c, build_model_d, build_model_e, build_model_f,
    build_yolov8_esi_full, C2fWithSE,
    print_model_comparison
)
from ultralytics.models.yolo.detect.train import DetectionTrainer

print_model_comparison()
print('\n✓ All custom modules loaded')

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

SSS DEBRIS DETECTION — MODEL VARIANTS
Variant    Name                           Backbone     Train     
--------------------------------------------------------------------------------
A          YOLOv8n baseline               COCO         Finetune  
B          YOLOv8s baseline               COCO         Finetune  
C          SS-YOLO scratch                Random       Scratch   
D          SS-YOLO pretrained             COCO         Finetune  
E          SS-YOLO+EIS scratch            Random       Scratch   
F          SS-YOLO pretrained+EIS         COCO         Finetune  

Key differences:
  A/B: Standard YOLO — easy to train, good transfer from COCO
  C/E: From scratch — needs 

In [ ]:
import pandas as pd

def evaluate(model_path, data_yaml, imgsz=256, conf=0.05):
    """Evaluate model and return metrics."""
    m = YOLO(model_path)
    r = m.val(data=data_yaml, imgsz=imgsz, conf=conf, verbose=False)
    p, rv = r.box.mp, r.box.mr
    f1 = 2*p*rv / max(p+rv, 1e-8)
    return {'mAP50': r.box.map50, 'P': p, 'R': rv, 'F1': f1}

def conf_sweep(model_path, data_yaml, imgsz=256):
    """Find optimal confidence threshold."""
    m = YOLO(model_path)
    best_f1, best_conf = 0, 0.05
    for c in [0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]:
        r = m.val(data=data_yaml, imgsz=imgsz, conf=c, verbose=False)
        p, rv = r.box.mp, r.box.mr
        f1 = 2*p*rv / max(p+rv, 1e-8)
        if f1 > best_f1:
            best_f1, best_conf = f1, c
    return best_conf, best_f1

def patch_trainer(model_obj):
    """Patch DetectionTrainer to use custom model."""
    _orig = DetectionTrainer.get_model
    def _patched(self, cfg=None, weights=None, verbose=True):
        from ultralytics.nn.tasks import DetectionModel
        from ultralytics.utils import RANK
        dm = DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'],
                            verbose=verbose and RANK == -1)
        dm.model = model_obj.model
        dm.nc = 1
        dm.names = {0: 'marine_debris'}
        try:
            dm.load(weights)
        except Exception:
            pass
        return dm
    DetectionTrainer.get_model = _patched
    return _orig

print('✓ Helpers loaded')

✓ Helpers loaded


In [ ]:
print('='*60)
print('STAGE 1: Model A — YOLOv8n baseline')
print('='*60)

model_a = YOLO('yolov8n.pt')
model_a.train(
    data=DATA, epochs=30, imgsz=256, batch=32, patience=15,
    lr0=0.01, lrf=0.01, warmup_epochs=2,
    mosaic=0.0, mixup=0.0,
    fliplr=0.0, flipud=0.0, degrees=0.0,
    translate=0.05, scale=0.2,
    name='model_a_yolov8n', project='/content/runs', exist_ok=True, plots=True,
)
print('✓ Model A done')

STAGE 1: Model A — YOLOv8n baseline
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/h8/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=model

In [ ]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.block import C2f

# --- GhostConv ---
class GhostConv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, p=0, r=2, dw=3):
        super().__init__()
        c = c2 // r
        self.primary = nn.Sequential(nn.Conv2d(c1, c, k, s, p, bias=False), nn.BatchNorm2d(c), nn.SiLU())
        self.cheap = nn.Sequential(nn.Conv2d(c, c, dw, 1, dw//2, groups=c, bias=False), nn.BatchNorm2d(c), nn.SiLU())
        self.n = c2 - c
    def forward(self, x):
        p = self.primary(x)
        return torch.cat([p, self.cheap(p)], dim=1)[:, :p.size(1)+self.n]

# --- FasterBlock (PConv + PWConv) ---
class FasterBlock(nn.Module):
    def __init__(self, c, shortcut=True):
        super().__init__()
        d = c // 4
        self.pconv = nn.Conv2d(d, d, 3, 1, 1, bias=False)
        self.ln1 = nn.LayerNorm(d)
        self.pw1 = nn.Conv2d(d, d*4, 1, bias=False)
        self.pw2 = nn.Conv2d(d*4, d, 1, bias=False) # Changed from c to d for channel compatibility
        self.ln2 = nn.LayerNorm(d) # Changed from c to d for channel compatibility
        self.act = nn.GELU()
        self.shortcut = shortcut
        self.d = d
    def forward(self, x):
        res = x
        x1, x2 = x.split([self.d, x.size(1)-self.d], dim=1)
        x1 = self.pconv(x1)
        x1 = self.ln1(x1.permute(0,2,3,1)).permute(0,3,1,2)
        x1 = self.act(self.pw1(x1))
        x1 = self.pw2(x1)
        x1 = self.ln2(x1.permute(0,2,3,1)).permute(0,3,1,2)
        return torch.cat([x1, x2], dim=1) + res if self.shortcut else torch.cat([x1, x2], dim=1)

# --- FastC2f ---
class FastC2f(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True):
        super().__init__()
        h = c2 // 2
        self.cv1 = nn.Conv2d(c1, h, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(h)
        self.blocks = nn.ModuleList([FasterBlock(h, shortcut) for _ in range(n)])
        self.cv2 = nn.Conv2d((n+2)*h//2, c2, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
    def forward(self, x):
        x = self.act(self.bn1(self.cv1(x)))
        x1, x2 = x.chunk(2, dim=1)
        y = [x1, x2]
        for b in self.blocks:
            y.append(b(y[-1]))
        return self.act(self.bn2(self.cv2(torch.cat(y, dim=1))))

# --- SE Block ---
class SEBlock(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(c, c//r, bias=False), nn.ReLU(), nn.Linear(c//r, c, bias=False), nn.Sigmoid())
    def forward(self, x):
        b,c,_,_ = x.size()
        w = self.fc(self.pool(x).view(b,c)).view(b,c,1,1)
        return x * w

# --- Build SS-YOLO (pretrained backbone, GhostConv+FastC2f neck) ---
def build_ss_yolo():
    yolo = YOLO('yolov8n.pt')
    layers = list(yolo.model.model)  # Get the Sequential
    for i, layer in enumerate(layers):
        f = getattr(layer, 'f', -1)
        if i <= 9:  # Keep backbone
            continue
        if isinstance(layer, Conv):
            c = layer.conv
            nl = GhostConv(c.in_channels, c.out_channels, c.kernel_size[0], c.stride[0], c.padding[0])
            nl.i, nl.f, nl.type = i, f, 'GhostConv'
            layers[i] = nl
        elif isinstance(layer, C2f):
            c1 = layer.cv1.conv.in_channels
            c2 = layer.cv2.conv.out_channels
            n = len(layer.m)
            nl = FastC2f(c1, c2, n=n)
            nl.i, nl.f, nl.type = i, f, 'FastC2f'
            layers[i] = nl
    yolo.model.model = nn.Sequential(*layers)
    total = sum(p.numel() for p in yolo.model.parameters())
    print(f'SS-YOLO built: {total/1e6:.2f}M params')
    return yolo.model

# --- Build YOLOv8-ESI (pretrained + SE attention) ---
def build_esi():
    yolo = YOLO('yolov8n.pt')
    layers = list(yolo.model.model)
    for i, layer in enumerate(layers):
        if isinstance(layer, C2f):
            c2 = layer.cv2.conv.out_channels
            se = SEBlock(c2)
            se.i, se.f, se.type = i, getattr(layer,'f',-1), 'SEBlock'
            # Wrap: run C2f then SE
            class C2fSE(nn.Module):
                def __init__(self, c2f, se):
                    super().__init__()
                    self.c2f = c2f
                    self.se = se
                    self.i = c2f.i
                    self.f = c2f.f
                def forward(self, x):
                    return self.se(self.c2f(x))
            layers[i] = C2fSE(layer, se)
    yolo.model.model = nn.Sequential(*layers)
    total = sum(p.numel() for p in yolo.model.parameters())
    print(f'YOLOv8-ESI built: {total/1e6:.2f}M params')
    return yolo.model

print('✓ Custom modules defined')

✓ Custom modules defined


In [ ]:
print('='*60)
print('STAGE 1: SS-YOLO (GhostConv + FastC2f neck)')
print('='*60)

from ultralytics.models.yolo.detect.train import DetectionTrainer
ss_model = build_ss_yolo()
_orig = DetectionTrainer.get_model
def _patch(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    dm = DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'], verbose=verbose and RANK==-1)
    dm.model = ss_model.model  # Corrected: Access the nn.Sequential module within the DetectionModel wrapper
    dm.nc = 1
    dm.names = {0: 'marine_debris'}
    try: dm.load(weights)
    except: pass
    return dm
DetectionTrainer.get_model = _patch
try:
    yolo_d = YOLO('yolov8n.pt')
    yolo_d.train(
        data=DATA, epochs=30, imgsz=256, batch=32, patience=15,
        lr0=0.01, lrf=0.01, warmup_epochs=2,
        freeze=10,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='model_d_ss_yolo', project='/content/runs', exist_ok=True, plots=True,
    )
finally:
    DetectionTrainer.get_model = _orig
print('✓ SS-YOLO done')

STAGE 1: SS-YOLO (GhostConv + FastC2f neck)
SS-YOLO built: 2.46M params
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/h8/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosai

In [ ]:
print('='*60)
print('STAGE 1: YOLOv8-ESI (SE attention)')
print('='*60)

esi_obj = build_yolov8_esi_full()
_orig2 = patch_trainer(esi_obj)

try:
    yolo_esi = YOLO('yolov8n.pt')
    yolo_esi.train(
        data=DATA, epochs=30, imgsz=256, batch=32, patience=15,
        lr0=0.01, lrf=0.01, warmup_epochs=2,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='model_esi', project='/content/runs', exist_ok=True, plots=True,
    )
finally:
    DetectionTrainer.get_model = _orig2

print('✓ YOLOv8-ESI done')

STAGE 1: YOLOv8-ESI (SE attention)
Building YOLOv8-ESI with SE attention blocks...
  (Pretrained YOLOv8n weights preserved — only SE layers are new)
  Wrapped layer 2: C2f(32) → C2fWithSE
  Wrapped layer 4: C2f(64) → C2fWithSE
  Wrapped layer 6: C2f(128) → C2fWithSE
  Wrapped layer 8: C2f(256) → C2fWithSE
  Wrapped layer 12: C2f(128) → C2fWithSE
  Wrapped layer 15: C2f(64) → C2fWithSE
  Wrapped layer 18: C2f(128) → C2fWithSE
  Wrapped layer 21: C2f(256) → C2fWithSE

YOLOv8-ESI (SE-augmented) built:
  Parameters: 3,180,880 (3.18M)
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/h8/data.yaml, degrees=0.0, determinist

In [ ]:
print('='*60)
print('STAGE 1: COMPARISON RESULTS')
print('='*60)

models_to_eval = [
    ('YOLOv8n', '/content/runs/model_a_yolov8n/weights/best.pt'),
    ('SS-YOLO-D', '/content/runs/model_d_ss_yolo/weights/best.pt'),
    ('YOLOv8-ESI', '/content/runs/model_esi/weights/best.pt'),
]

results = []
for name, path in models_to_eval:
    try:
        r = evaluate(path, DATA, conf=0.05)
        results.append({'Model': name, **r})
        print(f'  {name}: mAP50={r["mAP50"]:.4f}, P={r["P"]:.4f}, R={r["R"]:.4f}, F1={r["F1"]:.4f}')
    except Exception as e:
        print(f'  {name}: FAILED — {e}')

df = pd.DataFrame(results)
print('\n' + df.to_string(index=False))

if len(results) > 0:
    winner = max(results, key=lambda x: x['F1'])
    print(f'\n🏆 WINNER: {winner["Model"]} (F1={winner["F1"]:.4f})')
    winner_name = winner['Model']


STAGE 1: COMPARISON RESULTS
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4477.3±836.0 MB/s, size: 184.6 KB)
val: Scanning /content/h8/labels/val.cache... 438 images, 366 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 438/438 153.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 28/28 11.4it/s 2.5s
                   all        438         86      0.766      0.698       0.76      0.479
Speed: 0.5ms preprocess, 2.1ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/sonarvision/runs/detect/val
  YOLOv8n: mAP50=0.7602, P=0.7656, R=0.6977, F1=0.7301
Ultralytics 8.4.131 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary: 110 layers, 2,458,528 parameters, 0 gradients, 7.4 GFLOPs
val: Fast image access ✅ (ping